# DeepFaceLive — GPU (Google Colab) 🚀

Run your face-swap app on a **free Colab GPU** so the celebrity deepfake is **smooth**, not laggy.

### How to use (about 3 minutes)
1. Top menu: **Runtime → Change runtime type → T4 GPU → Save**.
2. Run each cell below **in order** — click the ▶ button on the left of each cell and wait for it to finish.
3. When the **last cell** prints a link ending in **`.trycloudflare.com`**, open **that link on your phone**.
4. Pick a celebrity model → **Load Model** → **Start Camera**.

> Notes: the first model you load downloads a file (~100 MB), so give it a few seconds. Keep this Colab tab open while you use the app — when you close it, the link stops working (that is normal, just re-run the cells next time).


In [ ]:
#@title 1) Check the GPU is switched on
# If this shows no GPU: Runtime -> Change runtime type -> T4 GPU -> Save, then re-run.
!nvidia-smi -L || echo "No GPU found — set Runtime -> Change runtime type -> T4 GPU, then re-run this cell."


In [ ]:
#@title 2) Download the app & install (about 2-3 min)
%cd /content
!rm -rf Deepfaketrial
!git clone --depth 1 -b colab-gpu https://github.com/oluwacoded/Deepfaketrial.git
%cd /content/Deepfaketrial

# Install the GPU build of onnxruntime (this is what makes it fast) plus the web deps
!pip -q uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null
!pip -q install onnxruntime-gpu flask flask-socketio eventlet numexpr h5py onnx

# cloudflared gives us a public https link the phone can open
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import onnxruntime as ort
provs = ort.get_available_providers()
print("onnxruntime providers:", provs)
print("GPU is ready for inference" if "CUDAExecutionProvider" in provs
      else "CUDA provider missing — make sure the runtime is set to GPU, then re-run this cell.")


In [ ]:
#@title 3) Start the app — then open the printed link on your PHONE
import subprocess, threading, re, time, sys, os

os.chdir('/content/Deepfaketrial')

def _drain(proc, tag):
    for line in proc.stdout:
        print(tag, line, end='')

# 1) start the face-swap server on port 5000 (it uses the GPU automatically)
server = subprocess.Popen([sys.executable, 'web_server.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
threading.Thread(target=_drain, args=(server, '[server]'), daemon=True).start()

print('Starting the face-swap server (loading the model on the GPU)...')
time.sleep(12)

# 2) open a public https tunnel to it
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
deadline = time.time() + 40
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break
# keep draining tunnel output so it does not stall
threading.Thread(target=_drain, args=(tunnel, '[tunnel]'), daemon=True).start()

print('\n' + '=' * 64)
if public_url:
    print('  OPEN THIS LINK ON YOUR PHONE:')
    print('      ' + public_url)
else:
    print('  Could not read the link automatically.')
    print('  Look in the [tunnel] lines above for a .trycloudflare.com link,')
    print('  or just re-run this cell.')
print('=' * 64)
print('Keep this Colab tab open while you use the app.')
